In [ ]:
folder = 'exp_23Jul27-1136'

In [ ]:
# load
import pickle
import pathlib
import pandas as pd

folder = pathlib.Path(folder)
assert folder.exists()

# aggregate results into csv necessary
file_list = list(folder.glob('result*.p'))
dict_list = list()
for file in file_list:
    with open(file, 'rb') as f:
        pval, seed, method, f1, sens, spec = pickle.load(file=f)
        
    dict_list.append({'pval': pval,
                      'seed': seed,
                      'method': method,
                      'f1': f1,
                      'sens': sens,
                      'spec': spec})
    
# load aggregated results
f_csv = folder / 'results.csv'
if f_csv.exists():
    df = pd.read_csv(f_csv, index_col=None)
else:
    df = pd.DataFrame()

# add in existing result
df = pd.concat((df, pd.DataFrame(dict_list)))

# round pvalue to 14 decimal places (avoids floating point comparison failure)
df['pval'] = df['pval'].round(14)

# drop duplicates & check for conflicting results
df.drop_duplicates(inplace=True)
assert df.value_counts(subset=['pval', 'seed', 'method']).max() == 1
    
# overwrite csv with latest / greatest
df.to_csv(f_csv, index=False)

# delete pickle files (they're in csv)
for file in file_list:
    file.unlink()

In [ ]:
import numpy as np
from collections import defaultdict


# extract
pval_list = sorted(df['pval'].unique())
seed_list = sorted(df['seed'].unique())

shape = len(seed_list), len(pval_list)
score_dict = defaultdict(lambda: np.full(shape=shape, fill_value=np.nan))

for _, row in df.iterrows():
    seed_idx = seed_list.index(row['seed'])
    pval_idx = pval_list.index(row['pval'])
    
    for feat in ('f1', 'sens', 'spec'):
        score_dict[row['method'], feat][seed_idx, pval_idx] = row[feat]

In [ ]:
# plot
import seaborn as sns
import matplotlib.pyplot as plt

sns.set()

fig, ax = plt.subplots(2, 3)

# plot top row
style_dict = {'AnalysisTFCE': {'color': 'r'}, 
              'AnalysisHRBA': {'color': 'b'}}
style_single = {'linewidth': .5,
                'zorder': 1,
                'label': '_nolegend_'}
style_mean = {'linewidth': 5,
              'zorder': 2,
              'label': '_nolegend_'}
for _ax, feat in zip(ax[0, :], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    for method, kwargs in style_dict.items():
        plt.plot(pval_list, score_dict[method, feat].T, **kwargs, **style_single)
        plt.plot(pval_list, np.nanmean(score_dict[method, feat], axis=0), **kwargs, **style_mean)

    plt.xlabel('pval')
    plt.ylabel(feat)
    plt.xscale('log')

# plot bottom row
for _ax, feat in zip(ax[1, :], ('f1', 'sens', 'spec')):
    plt.sca(_ax)
    x = score_dict['AnalysisHRBA', feat] - score_dict['AnalysisTFCE', feat]
    plt.axhline([0], linewidth=2, color='k')
    plt.plot(pval_list, x.T, color='k', **style_single)
    plt.plot(pval_list, np.nanmean(x, axis=0), color='k', **style_mean)

    plt.xlabel('pval')
    plt.ylabel(f'{feat}: HRBA - TFCE')
    plt.xscale('log')
    
# add legend in last plot of top row
plt.sca(ax[0, -1])
del style_single['label']
for method, kwargs in style_dict.items():
    plt.plot([], [], label=method[-4:], **kwargs, **style_single)
plt.legend()
    
fig.set_size_inches(10, 6)
fig.tight_layout()
fig.savefig(folder / 'HRBA_vs_TFCE.png', bbox_inches='tight')

# Examine Worst Cases
Where HRBA does poorest as compared to TFCE

In [ ]:
# import hrba.plot

# %matplotlib tk
# diff = score_dict['AnalysisHRBA', 'f1'] - score_dict['AnalysisTFCE', 'f1']

# for idx in np.argsort(diff.flatten())[:1]:
#     # lookup analysis & effect
#     seed, pval_idx = np.unravel_index(idx, diff.shape)
#     pval = pval_list[pval_idx]
#     ana, effect = pval_seed_method_ana_effect[pval][seed]['AnalysisHRBA']

#     # lookup / print f1 scores
#     f1_hrba = score_dict['AnalysisHRBA', 'f1'][seed, pval_idx]
#     f1_tfce = score_dict['AnalysisTFCE', 'f1'][seed, pval_idx]
#     print(f'\n\nseed {seed} pval: {pval:.2E} HRBA f1: {f1_hrba:.3f} TFCE f1: {f1_tfce:.3f}')
    
#     # plot
#     epoch = ana.epoch_list[0]
#     fig = hrba.plot.scatter_summary(epoch=epoch, mask=effect.mask)
#     fig.set_size_inches(8, 15)
#     plt.tight_layout()
#     plt.show()